# Original-chain ranks and ESS efficiency

Run `just diagnostic-plots` from the repository root. The Rust example exports all
ranks, R-hat components, ACF/integrated time, ESS, MCSE, and blocked mean errors.
This notebook validates and renders that self-contained report; it implements no
ESS or MCSE estimator. Outputs go under `target/notebooks/diagnostics/`.

The five fixed-seed scenarios show ordinary normal sampling, deliberate location
and scale disagreement, a narrow random walk with slow mixing, and a tied count
observable. Shifted/rescaled chains are constructed counterexamples, not samples
from a common target. These figures do not certify convergence or target correctness.

Every prefix was reranked in Rust. Plot ranks include all original prefix draws;
split diagnostics use first/last halves and omit one middle draw when the prefix
length is odd. Only complete, measured sampling workloads receive ESS/second.


In [ ]:
import hashlib
import json
import math
import os
from importlib.metadata import version
from pathlib import Path
from typing import TYPE_CHECKING

import matplotlib as mpl
import numpy as np
import polars as pl

if TYPE_CHECKING:
    from typing import Any

mpl.use("Agg")
import matplotlib.pyplot as plt

repo_root = Path(os.environ.get("MCMC_REPO_ROOT", ".")).resolve()
report_path = Path(os.environ.get("MCMC_DIAGNOSTICS_PATH", str(repo_root / "target/diagnostics.json"))).resolve()
output_dir = Path(os.environ.get("MCMC_NOTEBOOK_OUTPUT_DIR", str(repo_root / "target/notebooks/diagnostics"))).resolve()
report_bytes = report_path.read_bytes()
report = json.loads(report_bytes)

In [ ]:
def finite_number(value: Any) -> bool:
    """Reject booleans and nonfinite JSON numbers before plotting."""
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def validate_metrics(value: Any, field: str = "") -> None:
    """Preserve missing estimates and reasons throughout the exported tree."""
    if isinstance(value, dict):
        scalar_fields = {
            "estimate",
            "rate",
            "mean_mcse",
            "pooled_sample_sd",
            "mcse",
            "classical",
            "rank_normalized",
            "folded",
            "combined",
            "integrated_time",
            "mean_ess",
            "mean_ess_rate",
            "standard_error",
        }
        if field in scalar_fields or "value" in value:
            if "status" not in value or "value" not in value:
                message = "Scalar metric requires status and value"
                raise ValueError(message)
            valid = (
                value["status"] == "estimated"
                and finite_number(value["value"])
                and (value["value"] > 0 or (value["value"] == 0 and field in {"standard_error", "pooled_sample_sd"}))
            ) or (value["status"] == "unavailable" and value["value"] is None and bool(value.get("reason")))
            if not valid:
                message = "Diagnostic availability/value/reason mismatch"
                raise ValueError(message)
        for name, child in value.items():
            validate_metrics(child, name)
    elif isinstance(value, list):
        for child in value:
            validate_metrics(child)


def validate_acf(acf: dict[str, Any], length: int) -> None:
    """Check ACF availability and its finite, normalized lag sequence."""
    values = acf.get("values")
    if acf.get("status") == "estimated":
        valid = isinstance(values, list) and 0 < len(values) <= length and all(finite_number(value) and -1 <= value <= 1 for value in values) and values[0] == 1
    else:
        valid = (
            acf.get("status") == "unavailable"
            and values in (None, [])
            and acf.get("value") is None
            and isinstance(acf.get("reason"), str)
            and bool(acf["reason"])
        )
    if not valid:
        message = "Invalid ACF availability or normalized values"
        raise ValueError(message)


def validate_prefix(prefix: dict[str, Any], chains: list[dict[str, Any]]) -> None:
    """Check original identities, rank ranges, split counts, and timing scope."""
    count = len(chains)
    length = prefix["samples_per_chain"]
    full_length = len(chains[0]["draws"])
    if not isinstance(length, int) or isinstance(length, bool) or not 4 <= length <= full_length:
        message = "Invalid prefix length"
        raise ValueError(message)
    if (prefix["chain_count"], prefix["original_draws"], prefix["used_split_draws"], prefix["samples_per_split_chain"], prefix["omitted_middle_per_chain"]) != (
        count,
        count * length,
        count * 2 * (length // 2),
        length // 2,
        length % 2,
    ):
        message = "Prefix original/split counts disagree"
        raise ValueError(message)
    chain_ids = [chain["chain_id"] for chain in chains]
    if any([row["chain_id"] for row in prefix[field]] != chain_ids for field in ("ranks", "single_chain")):
        message = "Diagnostic rows do not preserve original chain identity"
        raise ValueError(message)
    for row in prefix["ranks"]:
        if len(row["ranks"]) != length or any(not finite_number(rank) or not 1 <= rank <= count * length for rank in row["ranks"]):
            message = "Rank row does not match the selected original prefix"
            raise ValueError(message)
    for row in prefix["single_chain"]:
        validate_acf(row["acf"], length)
    validate_timing(prefix, chains)
    timing = prefix["timing"]
    for name in ("mean", "bulk", "tail"):
        estimate = prefix["ess"][name]
        value = estimate["estimate"]["value"]
        if value is None:
            if estimate["relative"] is not None or estimate["rate"]["value"] is not None:
                message = "Unavailable ESS acquired a relative value or rate"
                raise ValueError(message)
        elif not finite_number(estimate["relative"]) or not math.isclose(estimate["relative"], value / prefix["used_split_draws"], rel_tol=1e-12):
            message = "Relative ESS uses the wrong retained count"
            raise ValueError(message)
        rate = estimate["rate"]
        if rate["status"] == "estimated" and (
            timing["status"] != "measured" or not math.isclose(rate["value"], value / timing["elapsed_seconds"], rel_tol=1e-12)
        ):
            message = "ESS rate requires matching measured timing"
            raise ValueError(message)
    validate_metrics(prefix)


def validate_timing(prefix: dict[str, Any], chains: list[dict[str, Any]]) -> None:
    """Accept measured time only for the complete, matching sequential run."""
    length = prefix["samples_per_chain"]
    full_length = len(chains[0]["draws"])
    timing = prefix["timing"]
    if timing["status"] == "measured":
        elapsed = [chain["elapsed_seconds"] for chain in chains]
        if length != full_length or not all(finite_number(seconds) and seconds > 0 for seconds in elapsed):
            message = "Measured timing requires the complete original workload"
            raise ValueError(message)
        if not finite_number(timing["elapsed_seconds"]) or not math.isclose(timing["elapsed_seconds"], sum(elapsed), rel_tol=1e-12):
            message = "Prefix timing disagrees with sequential chain durations"
            raise ValueError(message)
    elif timing["status"] != "unavailable" or timing["elapsed_seconds"] is not None or not timing["reason"]:
        message = "Invalid unavailable timing"
        raise ValueError(message)

In [ ]:
if report["schema_version"] != 2 or report["workflow"] != "rank_ess_efficiency_v1":
    message = "Expected diagnostics schema 2 / rank_ess_efficiency_v1"
    raise ValueError(message)
if report["rank_convention"] != "one_based_average_ties_all_original_prefix_draws_signed_zero_equal":
    message = "Unsupported rank/draw selection convention"
    raise ValueError(message)
if report["split_convention"] != "first_and_last_floor_N_over_2" or report["quantile_convention"] != "linear_type_7_all_original_prefix_draws":
    message = "Unsupported split or quantile convention"
    raise ValueError(message)
if report["timing_scope"] != "sequential_production_sampling_and_recording_excluding_allocation_warmup_analysis_export":
    message = "Unsupported production timing scope"
    raise ValueError(message)
runs = report["runs"]
if not runs or len({run["run_id"] for run in runs}) != len(runs):
    message = "Report requires uniquely identified runs"
    raise ValueError(message)
for run in runs:
    cadence = run["recording_interval"]
    if not isinstance(cadence, int) or isinstance(cadence, bool) or cadence <= 0:
        message = "Recording interval must be a positive integer"
        raise ValueError(message)
    chains = run["chains"]
    if len(chains) < 2 or len({chain["chain_id"] for chain in chains}) != len(chains) or run["chain_identity"] != "original_unsplit":
        message = "Report requires distinct original chains"
        raise ValueError(message)
    lengths = [len(chain["draws"]) for chain in chains]
    if len(set(lengths)) != 1 or min(lengths) < 4 or any(not finite_number(value) for chain in chains for value in chain["draws"]):
        message = "Original chains must have equal lengths and finite draws"
        raise ValueError(message)
    prefixes = [prefix["samples_per_chain"] for prefix in run["prefixes"]]
    if not prefixes or prefixes != report["prefixes"] or prefixes != sorted(set(prefixes)) or prefixes[-1] != lengths[0]:
        message = "Prefix inventory must increase through the complete run"
        raise ValueError(message)
    for prefix in run["prefixes"]:
        validate_prefix(prefix, chains)

output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "diagnostics.json").write_bytes(report_bytes)
print(f"Validated {len(runs)} runs; preserved exact Rust evidence in {output_dir}")

## Compare original chains against the pooled bins

The solid curves below show each original chain's fraction of draws per bin.
The dashed curve is the fraction in the pooled bin, using the same edges.
Ties retain deterministic average ranks. For discrete/count observations, empty
bins and peaks can occur even when chains agree; the pooled reference makes that
structure visible. There is no uniformity test or automatic pass/fail threshold.

Set `rank_prefix` below to one of the exported prefix sizes to inspect a shorter
run. Its ranks were recomputed from that prefix's observations.


In [ ]:
rank_prefix = report["prefixes"][-1]
if rank_prefix not in report["prefixes"]:
    message = "Rank prefix must be present in the Rust report"
    raise ValueError(message)
rank_rows = []
rank_figure, rank_axes = plt.subplots(len(runs), 1, figsize=(11, 3.1 * len(runs)), squeeze=False, layout="constrained")
for run, axis in zip(runs, rank_axes[:, 0], strict=True):
    prefix = next(prefix for prefix in run["prefixes"] if prefix["samples_per_chain"] == rank_prefix)
    edges = np.linspace(0.5, prefix["original_draws"] + 0.5, 21)
    rank_series = [(row["chain_id"], row["ranks"]) for row in prefix["ranks"]]
    pooled = [rank for _, ranks in rank_series for rank in ranks]
    for chain_id, ranks in [*rank_series, ("pooled reference", pooled)]:
        counts, _ = np.histogram(ranks, bins=edges)
        proportions = counts / len(ranks)
        style = {"color": "black", "linestyle": "--", "linewidth": 2} if chain_id == "pooled reference" else {}
        axis.stairs(proportions, edges, baseline=None, label=chain_id, **style)
        rank_rows.extend(
            {
                "run_id": run["run_id"],
                "observable": run["observable"],
                "original_prefix_per_chain": rank_prefix,
                "chain_id": chain_id,
                "bin_left": float(left),
                "bin_right": float(right),
                "count": int(count),
                "total": len(ranks),
                "proportion": float(proportion),
            }
            for left, right, count, proportion in zip(edges[:-1], edges[1:], counts, proportions, strict=True)
        )
    axis.set(
        title=f"{run['scenario'].replace('_', ' ')} | {run['observable']} | N={rank_prefix} per original chain",
        xlabel="Pooled average rank (all original prefix draws)",
        ylabel="Fraction of draws in bin",
    )
    axis.legend(ncols=5, fontsize=8)
rank_figure.savefig(output_dir / "rank_overlays.png", dpi=140)
plt.close(rank_figure)
pl.DataFrame(rank_rows).write_csv(output_dir / "rank_bins.csv")

## Information, uncertainty, and elapsed time are different quantities

ESS is an effective sample count for the named estimator. Relative ESS is ESS/S,
where S is the total **retained split draw count**; it can exceed one for
antithetic draws. ESS/second is a measured workload rate, absent for these untimed
prefixes and post-processing transformations. Missing estimates stay missing.

The pooled sample standard deviation describes observed spread. Mean/quantile
MCSE describes uncertainty in an estimated summary, in the observable's original
units. Blocked mean errors have a separate per-chain contract; their block sizes
and used counts are retained rather than choosing a favorable level.


In [ ]:
efficiency_rows = []
efficiency_figure, efficiency_axes = plt.subplots(len(runs), 2, figsize=(12, 3 * len(runs)), squeeze=False, layout="constrained")
for run, axes in zip(runs, efficiency_axes, strict=True):
    retained = [prefix["used_split_draws"] for prefix in run["prefixes"]]
    for name in ("mean", "bulk", "tail"):
        for prefix in run["prefixes"]:
            metric = prefix["ess"][name]
            efficiency_rows.append(
                {
                    "run_id": run["run_id"],
                    "observable": run["observable"],
                    "estimator": name,
                    "original_prefix_per_chain": prefix["samples_per_chain"],
                    "original_draws": prefix["original_draws"],
                    "used_split_draws": prefix["used_split_draws"],
                    "ess": metric["estimate"]["value"],
                    "status": metric["estimate"]["status"],
                    "reason": metric["estimate"].get("reason"),
                    "relative_ess": metric["relative"],
                    "ess_per_second": metric["rate"]["value"],
                    "rate_status": metric["rate"]["status"],
                    "rate_reason": metric["rate"].get("reason"),
                }
            )
        if name == "mean":
            continue
        for axis, key in zip(axes, ("estimate", "relative"), strict=True):
            values = [prefix["ess"][name][key] for prefix in run["prefixes"]]
            values = [value["value"] if isinstance(value, dict) else value for value in values]
            label = f"{name} (some prefixes unavailable)" if any(value is None for value in values) else name
            axis.plot(retained, [np.nan if value is None else value for value in values], marker="o", label=label)
    for axis, label in zip(axes, ("ESS (effective draws)", "Relative ESS = ESS/S"), strict=True):
        axis.set(title=run["scenario"].replace("_", " "), xlabel="S: retained draws across split chains", ylabel=label)
        axis.legend()
        axis.grid(alpha=0.25)
efficiency_figure.savefig(output_dir / "ess_efficiency.png", dpi=140)
plt.close(efficiency_figure)
pl.DataFrame(efficiency_rows).write_csv(output_dir / "efficiency.csv")

In [ ]:
summary_rows = []
blocked_rows = []
for run in runs:
    full = run["prefixes"][-1]
    metrics = [
        ("pooled_sample_sd", full["pooled_sample_sd"], run["units"]),
        ("mean_mcse", full["mean_mcse"], run["units"]),
        *((f"rhat_{name}", metric, "dimensionless") for name, metric in full["rhat"].items()),
        *((f"quantile_mcse_p{quantile['probability']}", quantile["mcse"], run["units"]) for quantile in full["quantiles"]),
    ]
    for name, metric, units in metrics:
        summary_rows.append(
            {
                "run_id": run["run_id"],
                "observable": run["observable"],
                "metric": name,
                "units": units,
                "original_draws": full["original_draws"],
                "used_split_draws": full["used_split_draws"],
                "value": metric["value"],
                "status": metric["status"],
                "reason": metric.get("reason"),
            }
        )
    for chain in full["single_chain"]:
        blocked = chain["blocked_mean_error"]
        levels: list[dict[str, Any]] = (
            blocked.get("levels", [])
            if blocked["status"] == "estimated"
            else [{"block_size": None, "block_count": None, "used_draws": None, "standard_error": blocked}]
        )
        for level in levels:
            blocked_rows.append(
                {
                    "run_id": run["run_id"],
                    "observable": run["observable"],
                    "chain_id": chain["chain_id"],
                    "block_size": level["block_size"],
                    "block_count": level["block_count"],
                    "used_draws": level["used_draws"],
                    "standard_error": level["standard_error"]["value"],
                    "status": level["standard_error"]["status"],
                    "reason": level["standard_error"].get("reason"),
                    "units": run["units"],
                }
            )
summary = pl.DataFrame(summary_rows)
summary.write_csv(output_dir / "summary.csv")
pl.DataFrame(blocked_rows).write_csv(output_dir / "blocked_errors.csv")
with pl.Config(tbl_rows=50, tbl_cols=7):
    print(summary.select("run_id", "metric", "value", "units", "status", "reason"))

In [ ]:
trace_figure, trace_axes = plt.subplots(len(runs), 2, figsize=(12, 3 * len(runs)), squeeze=False, layout="constrained")
for run, axes in zip(runs, trace_axes, strict=True):
    full = run["prefixes"][-1]
    for chain, diagnostic in zip(run["chains"], full["single_chain"], strict=True):
        axes[0].plot(range(1, len(chain["draws"]) + 1), chain["draws"], linewidth=0.6, label=chain["chain_id"])
        if diagnostic["acf"]["status"] == "estimated":
            acf = diagnostic["acf"]["values"][:65]
            axes[1].plot(range(len(acf)), acf, label=chain["chain_id"])
        else:
            axes[1].plot([], [], linestyle="none", label=f"{chain['chain_id']}: ACF unavailable ({diagnostic['acf']['reason']})")
    axes[0].set(
        title=run["scenario"].replace("_", " "),
        xlabel=f"Recorded production draw (cadence = {run['recording_interval']})",
        ylabel=f"{run['observable']} ({run['units']})",
    )
    axes[1].set(title=run["scenario"].replace("_", " "), xlabel="Lag (recorded draws)", ylabel="Autocorrelation")
    for axis in axes:
        axis.legend(ncols=2, fontsize=8)
trace_figure.savefig(output_dir / "traces_acf.png", dpi=140)
plt.close(trace_figure)

manifest = {
    "schema_version": 1,
    "report_sha256": hashlib.sha256(report_bytes).hexdigest(),
    "report_copy": "diagnostics.json",
    "rank_prefix_per_chain": rank_prefix,
    "crate_version": report["crate_version"],
    "source_revision": report["source_revision"],
    "source_dirty": report["source_dirty"],
    "plotting_versions": {name: version(name) for name in ("matplotlib", "numpy", "polars")},
    "figures": ["rank_overlays.png", "ess_efficiency.png", "traces_acf.png"],
    "tables": ["rank_bins.csv", "efficiency.csv", "summary.csv", "blocked_errors.csv"],
}
(output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8", newline="\n")
print(f"Saved three figures, four CSV tables, and matching source evidence in {output_dir}")

The rank and efficiency presentations follow
[Vehtari et al. (2021), Section 4.5](https://doi.org/10.1214/20-BA1221).
Public Rust diagnostic contracts and the difference between original/split draws
are documented in [Analyzing chains](../docs/ANALYZING_CHAINS.md).
Independent verification uses pinned SciPy 1.16.2 average ranks and ArviZ 0.22.0
via `tests/fixtures/generate_diagnostic_plots.py --report target/diagnostics.json`.
The notebook is a rendering consumer, not a second numerical oracle.

Inspect each chain and observable. Agreement of ranks, large ESS, or small MCSE
does not establish exploration of every mode, valid target moments, or correct
sampling. The deliberately transformed runs violate common-target assumptions.
Select scientific tolerances and stopping decisions in the downstream application.
